In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

In [3]:
print(sys.path)

['/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/src', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages/setuptools/_vendor', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework']


In [4]:
root_path = (Path(os.getcwd()).parent.parent / "src").as_posix()
root_path

'/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework/src'

In [5]:
sys.path.append(root_path)

In [42]:
import unittest
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, Row, DecimalType
from adm.integration.adm_data_transformer import AdmDataTransformer
from adm.integration.adm_base_data_transformer import AdmBaseDataTransformer
from adm.integration.adm_table_writer import AdmTableWriter
from adm.integration.adm_table_reader import AdmTableReader
import logging

In [43]:
config_file = "use_cases/p2c_insightops/sales_order_invoice_ibmdb2.yml"
adm_table_reader = AdmTableReader(config_file)
transformer = AdmBaseDataTransformer(config_file)
transformer2 = AdmDataTransformer(config_file)
writer = AdmTableWriter(config_file)


In [44]:
df = adm_table_reader.read_source_table("ibmdb2_currencytable")
df = df.filter((F.col("FCURR") == "EUR") & (F.col("date_converted") == F.lit("2025-03-03")))
df.dtypes

INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_currencytable
INFO:adm.integration.base_integration:Applying function Name: use_cases.custom_transformations.p2c_tx.currency_convert_from_source_currency_to_target_currency_fun


[('FCURR', 'string'),
 ('TCURR', 'string'),
 ('date_converted', 'date'),
 ('exchange_rate', 'decimal(21,15)')]

In [45]:
display(df)

,FCURR,TCURR,date_converted,exchange_rate


In [48]:
df = transformer2.apply_joins("combine_1")

INFO:adm.integration.base_integration:Applying joins for combine_id: combine_1
INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_inv_ord_details
INFO:adm.integration.base_integration:Reading and selecting columns for source_id: ibmdb2_invoice_details with alias: id
INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_invoice_details
INFO:adm.integration.base_integration:Joining iod with id using condition: iod.INOR_ID = id.INVC_INOR_ID and join type: inner
INFO:adm.integration.base_integration:Schema after joining iod with id: StructType([StructField('INOR_ID', IntegerType(), False), StructField('INOR_COPC_ID', IntegerType(), False), StructField('INOR_TC_TERMS_CODE', StringType(), False), StructField('INOR_SOLDTO', StringType(), False), StructField('INOR_CURR_ID', IntegerType(), False), StructField('INOR_ORIG_ORDER_NUMBER', StringType(), False), StructField('INOR_SHIP_DATE', DateType(), True), StructField('INOR_ORDER_DATE', DateType(

In [49]:
#display(df.filter("CURR_CODE == 'USD'"))
display(df.filter(F.col("exchange_rate").isNotNull()))

,INOR_ID,INOR_COPC_ID,INOR_TC_TERMS_CODE,INOR_SOLDTO,INOR_CURR_ID,INOR_ORIG_ORDER_NUMBER,INOR_SHIP_DATE,INOR_ORDER_DATE,INOR_TOT_EXTENDED_AMT,INVC_INOR_ID,INVC_INVOICE_NBR,INVC_INVOICE_DATE,COPC_ID,COPC_CODE,COPC_CODE_int,COPC_NAME,TERM_CODE,TERM_DESC,VNCU_VENDOR_NBR,VNCU_NAME,VNCU_UPDATE_USERID,CURR_ID,CURR_CODE,OI_ORIG_ORDER,OI_TYPE,OI_ITEM_FROM,XRFC_COMPANY,XRFC_COPC,CD_COPC,CD_INTL_CREDIT_MGR,CMDG_COMD_GRP_CD,CMDG_CORP_SEGMENT_CD,PC_COPC,PC_GRP_COMMODITY,CSEG_SEGMENT_CD,CSEG_DESC,LOCATION_CODE,COUNTRY,date_converted,FCURR,TCURR,exchange_rate
0,25761858,3061,307,278956,177531,182426,2021-05-31,2021-05-31,2490.44,25761858,935058,2021-05-31,3061,4271,4271,ADM AGRI-IND LTD ST HYACINTHE,307,NET 30 JOURS,278956,H L BOISVERT INC,X486601,177531,CAD,182426,AI,OP SALES,4271,4271,4271,NAM,79,05,4271,79,05,NUTRITION,None,None,2021-05-31,CAD,USD,0.827700000000000
1,28083642,2790,613,425009,177553,115924,2023-03-09,2022-12-20,73260.00,28083642,130476,2023-03-09,2790,1010,1010,ADM PROCESSING SPECIALTY FOOD ING,613,90 DAY B/L,425009,LACTOTECNIA S L,#40995,177553,EUR,115924,AI,OP SALES,1010,1010,1010,NAM,17,04,1010,17,04,NUTRITION,None,None,2023-03-09,EUR,USD,1.055100000000000
2,26525127,2790,613,425009,177553,148107,2022-02-17,2021-11-30,87380.00,26525127,720130,2022-02-17,2790,1010,1010,ADM PROCESSING SPECIALTY FOOD ING,613,90 DAY B/L,425009,LACTOTECNIA S L,#40995,177553,EUR,148107,AI,OP SALES,1010,1010,1010,NAM,17,04,1010,17,04,NUTRITION,None,None,2022-02-17,EUR,USD,1.135500000000000
3,30568800,3061,864,289164,177531,339105,2024-09-16,2024-09-13,439.00,30568800,295827,2024-09-18,3061,4271,4271,ADM AGRI-IND LTD ST HYACINTHE,864,3 % 30 DAYS,289164,FERME M GRENIER ET FILS INC,#40995,177531,CAD,339105,AI,OP SALES,4271,4271,4271,NAM,79,05,4271,79,05,NUTRITION,None,None,2024-09-18,CAD,USD,0.735900000000000
4,25538965,3062,007,556732,177531,907580,2021-04-15,2021-04-07,30.45,25538965,921904,2021-04-16,3062,4272,4272,ADM AGRI-IND LTD LETHBRIDGE AB CD$,007,REMIT 30 DAYS,556732,HEARTLAND FEEDS/UEBELHARDT ENT LTD,#40995,177531,CAD,None,None,None,4272,4272,4272,NAM,79,05,4272,79,05,NUTRITION,None,None,2021-04-16,CAD,USD,0.798300000000000
5,25802347,2790,613,425009,177553,717404,2021-05-13,2021-06-10,-44664.00,25802347,439979,2021-06-10,2790,1010,1010,ADM PROCESSING SPECIALTY FOOD ING,613,90 DAY B/L,425009,LACTOTECNIA S L,#40995,177553,EUR,None,None,None,1010,1010,1010,NAM,17,04,1010,17,04,NUTRITION,None,None,2021-06-10,EUR,USD,1.216300000000000
6,25403977,3011,023,278956,177531,739599,2021-03-06,2021-03-05,47668.90,25403977,909427,2021-03-08,3011,4205,4205,ADM AGRI-IND LTD PROTEIN CD$,023,NET CASH REC INVOICE,278956,H L BOISVERT INC,X486601,177531,CAD,None,None,None,4205,4205,4205,NAM,02,01,4205,02,01,AG SERVICES & OILSEEDS,None,None,2021-03-08,CAD,USD,0.789800000000000
7,25871255,3061,307,278956,177531,324508,2021-06-26,2021-06-27,6089.70,25871255,943567,2021-06-29,3061,4271,4271,ADM AGRI-IND LTD ST HYACINTHE,307,NET 30 JOURS,278956,H L BOISVERT INC,X486601,177531,CAD,None,None,None,4271,4271,4271,NAM,79,05,4271,79,05,NUTRITION,None,None,2021-06-29,CAD,USD,0.810400000000000
8,25199741,3061,307,278956,177531,482199,2021-01-16,2021-01-17,4929.60,25199741,894527,2021-01-19,3061,4271,4271,ADM AGRI-IND LTD ST HYACINTHE,307,NET 30 JOURS,278956,H L BOISVERT INC,X486601,177531,CAD,None,None,None,4271,4271,4271,NAM,79,05,4271,79,05,NUTRITION,None,None,2021-01-19,CAD,USD,0.785700000000000
9,25788110,3061,307,278956,177531,215072,2021-06-05,2021-06-07,6916.50,25788110,937339,2021-06-08,3061,4271,4271,ADM AGRI-IND LTD ST HYACINTHE,307,NET 30 JOURS,278956,H L BOISVERT INC,X486601,177531,CAD,None,None,None,4271,4271,4271,NAM,79,05,4271,79,05,NUTRITION,None,None,2021-06-08,CAD,USD,0.827100000000000


In [50]:
df2 = transformer.apply_transformations("ibmdb2_joined", df)

INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_joined


In [53]:
display(df2.filter(F.col("exchange_rate").isNotNull()))

,shipment_date,order_date,sold_to_customer_no,sold_to_customer_name,invoice_number,total_invoice_amount_usd,invoice_date,payment_term_code,payment_term_description,entity_document_company_code,entity_document_company_name,invoice_due_date,user_id,source_system,currency_code,segment,region,document_type,gbs_group,exchange_rate,__inserted_on,__inserted_timestamp
0,2021-05-31,2021-05-31,278956,H L BOISVERT INC,935058,2061.337188,2021-05-31,307,NET 30 JOURS,4271,ADM AGRI-IND LTD ST HYACINTHE,None,X486601,ibmdb2,CAD,NUTRITION,NA,AI,N,0.827700000000000,2025-04-30,2025-04-30 19:18:49.141
1,2023-03-09,2022-12-20,425009,LACTOTECNIA S L,130476,77296.626000,2023-03-09,613,90 DAY B/L,1010,ADM PROCESSING SPECIALTY FOOD ING,None,#40995,ibmdb2,EUR,NUTRITION,NA,AI,N,1.055100000000000,2025-04-30,2025-04-30 19:18:49.141
2,2022-02-17,2021-11-30,425009,LACTOTECNIA S L,720130,99219.990000,2022-02-17,613,90 DAY B/L,1010,ADM PROCESSING SPECIALTY FOOD ING,None,#40995,ibmdb2,EUR,NUTRITION,NA,AI,N,1.135500000000000,2025-04-30,2025-04-30 19:18:49.141
3,2024-09-16,2024-09-13,289164,FERME M GRENIER ET FILS INC,295827,323.060100,2024-09-18,864,3 % 30 DAYS,4271,ADM AGRI-IND LTD ST HYACINTHE,None,#40995,ibmdb2,CAD,NUTRITION,NA,AI,N,0.735900000000000,2025-04-30,2025-04-30 19:18:49.141
4,2021-04-15,2021-04-07,556732,HEARTLAND FEEDS/UEBELHARDT ENT LTD,921904,24.308235,2021-04-16,007,REMIT 30 DAYS,4272,ADM AGRI-IND LTD LETHBRIDGE AB CD$,None,#40995,ibmdb2,CAD,NUTRITION,NA,None,N,0.798300000000000,2025-04-30,2025-04-30 19:18:49.141
5,2021-05-13,2021-06-10,425009,LACTOTECNIA S L,439979,-54324.823200,2021-06-10,613,90 DAY B/L,1010,ADM PROCESSING SPECIALTY FOOD ING,None,#40995,ibmdb2,EUR,NUTRITION,NA,None,N,1.216300000000000,2025-04-30,2025-04-30 19:18:49.141
6,2021-03-06,2021-03-05,278956,H L BOISVERT INC,909427,37648.897220,2021-03-08,023,NET CASH REC INVOICE,4205,ADM AGRI-IND LTD PROTEIN CD$,None,X486601,ibmdb2,CAD,AG SERVICES & OILSEEDS,NA,None,N,0.789800000000000,2025-04-30,2025-04-30 19:18:49.141
7,2021-06-26,2021-06-27,278956,H L BOISVERT INC,943567,4935.092880,2021-06-29,307,NET 30 JOURS,4271,ADM AGRI-IND LTD ST HYACINTHE,None,X486601,ibmdb2,CAD,NUTRITION,NA,None,N,0.810400000000000,2025-04-30,2025-04-30 19:18:49.141
8,2021-01-16,2021-01-17,278956,H L BOISVERT INC,894527,3873.186720,2021-01-19,307,NET 30 JOURS,4271,ADM AGRI-IND LTD ST HYACINTHE,None,X486601,ibmdb2,CAD,NUTRITION,NA,None,N,0.785700000000000,2025-04-30,2025-04-30 19:18:49.141
9,2021-06-05,2021-06-07,278956,H L BOISVERT INC,937339,5720.637150,2021-06-08,307,NET 30 JOURS,4271,ADM AGRI-IND LTD ST HYACINTHE,None,X486601,ibmdb2,CAD,NUTRITION,NA,None,N,0.827100000000000,2025-04-30,2025-04-30 19:18:49.141


In [52]:
df2 = transformer.apply_transformations("ibmdb2_joined", df)
display(df2)

INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_joined


,shipment_date,order_date,sold_to_customer_no,sold_to_customer_name,invoice_number,total_invoice_amount_usd,invoice_date,payment_term_code,payment_term_description,entity_document_company_code,entity_document_company_name,invoice_due_date,user_id,source_system,currency_code,segment,region,document_type,gbs_group,exchange_rate,__inserted_on,__inserted_timestamp
0,2014-12-01,2014-11-21,3729,RAND MATERIALS & HANDLING,629274,5183.20,2014-12-01,024,NET CASH,1018,ADM PROCESSING RED WING PROCESSING,None,A730805,ibmdb2,USD,AG SERVICES & OILSEEDS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
1,2014-11-25,2014-11-21,720157,SEBADE FARMS,418343,1371.00,2014-11-25,024,NET CASH,9503,ADM CORN PROCESSING FEED INGREDIENTS,None,#40995,ibmdb2,USD,CARBOHYDRATE SOLUTIONS,NA,AI,N,None,2025-04-30,2025-04-30 19:15:40.712
2,2007-06-01,2007-06-04,431560,MOLO OIL CO,020641,1852.67,2007-06-04,005,REMIT 10 DAYS,9517,ADM CORN PROCESSING POWER ALCOHOL,None,#40995,ibmdb2,USD,CARBOHYDRATE SOLUTIONS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
3,2006-09-14,2007-05-31,730401,NOBLE AMERICAS CORP,017395,33000.00,2007-05-31,023,NET CASH REC INVOICE,9517,ADM CORN PROCESSING POWER ALCOHOL,None,#40995,ibmdb2,USD,CARBOHYDRATE SOLUTIONS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
4,2014-11-24,2014-11-21,19159,BARNES RAILROAD SERVICES INC,413260,9856.89,2014-11-24,275,REMIT ON REC OF INV,9502,ADM CORN PROCESSING CORN SYRUP/DEXTROSE,None,MISSCH1,ibmdb2,USD,CARBOHYDRATE SOLUTIONS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
5,2014-11-24,2014-11-21,672122,PILLEN PRODUCTS CORP ACCT SOUTH,414081,1543.77,2014-11-24,024,NET CASH,9503,ADM CORN PROCESSING FEED INGREDIENTS,None,#40995,ibmdb2,USD,CARBOHYDRATE SOLUTIONS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
6,2014-11-22,2014-11-21,869860,LITTLE CREEK DAIRY,110338,4105.67,2014-11-24,012,REMIT REC INVOICE,2041,SOUTHERN COTTON OIL LUBBOCK,None,#40995,ibmdb2,USD,AG SERVICES & OILSEEDS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
7,2014-11-20,2014-11-21,856143,"LINCOLN ENERGY SOLUTIONS, LLC",412496,15850.55,2014-11-21,023,NET CASH REC INVOICE,9517,ADM CORN PROCESSING POWER ALCOHOL,None,#40995,ibmdb2,USD,CARBOHYDRATE SOLUTIONS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
8,2014-11-24,2014-11-21,862635,SUWANNEE VALLEY FEEDS LLC,623338,10247.50,2014-11-25,024,NET CASH,1049,ADM PROCESSING SB MEAL VALDOSTA,None,#40995,ibmdb2,USD,AG SERVICES & OILSEEDS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712
9,2014-11-25,2014-11-21,890134,CBJ FARMS,110683,7545.71,2014-11-25,012,REMIT REC INVOICE,2041,SOUTHERN COTTON OIL LUBBOCK,None,INTERNT,ibmdb2,USD,AG SERVICES & OILSEEDS,NA,None,N,None,2025-04-30,2025-04-30 19:15:40.712


In [8]:
df = adm_table_reader.read_source_table("ibmdb2_inv_ord_details")
df.dtypes

INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_inv_ord_details


[('INOR_ID', 'int'),
 ('INOR_COPC_ID', 'int'),
 ('INOR_TC_TERMS_CODE', 'string'),
 ('INOR_SOLDTO', 'string'),
 ('INOR_CURR_ID', 'int'),
 ('INOR_ORIG_ORDER_NUMBER', 'string'),
 ('INOR_SHIP_DATE', 'date'),
 ('INOR_ORDER_DATE', 'date'),
 ('INOR_TOT_EXTENDED_AMT', 'decimal(13,2)')]

In [80]:
df = adm_table_reader.read_source_table("ibmdb2_profit_center")
df.columns

['COPC_ID',
 'COPC_CODE',
 'COPC_NAME',
 'COPC_DELETED',
 'COPC_CONTGRP_ID',
 'COPC_PRODUCTGROUP_ID',
 'COPC_REP_GROUP_ID',
 'COPC_COMM_REPG_ID',
 'COPC_COMP_ID']

In [9]:
df = adm_table_reader.read_source_table("ibmdb2_invoice_details")
df.dtypes

INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_invoice_details


[('INVC_INOR_ID', 'int'),
 ('INVC_INVOICE_NBR', 'string'),
 ('INVC_INVOICE_DATE', 'date')]

In [30]:
df = adm_table_reader.read_source_table("ibmdb2_term_code")
df.columns

['TERM_ID',
 'TERM_CODE',
 'TERM_DESC',
 'TERM_ABBR_DESC',
 'TERM_SALE_ID',
 'TERM_ACTIVE_SW']

In [31]:
df = adm_table_reader.read_source_table("ibmdb2_vendor_customer")
df.columns

['VNCU_ID',
 'VNCU_VENDOR_NBR',
 'VNCU_NAME',
 'VNCU_PRIMARY_PHONE',
 'VNCU_ORIG_DT',
 'VNCU_LAST_ACTIVITY_DT',
 'VNCU_UPDATE_USERID',
 'VNCU_STAT_ID',
 'VNCU_CURR_ID',
 'VNCU_REFER_VENDOR_NBR',
 'VNCU_SINGLE_REMIT_IND',
 'VNCU_SORT_NAME',
 'VNCU_TAX_ID',
 'VNCU_PRIORITY_REMARKS',
 'VNCU_NOTIFY_COUNT',
 'VNCU_INDUSTRY_CODE',
 'VNCU_VAT_TAX_ID']

In [32]:
df = adm_table_reader.read_source_table("ibmdb2_open_item")
df.columns

['OI_SEQ_NBR',
 'OI_COPC',
 'OI_ACCT_NBR',
 'OI_ITEM_DATE',
 'OI_ITEM_NBR',
 'OI_CD_DUE_DATE',
 'OI_INV_AMT',
 'OI_CURR',
 'OI_TYPE',
 'OI_NET_DUE_DATE',
 'OI_US_AMT',
 'OI_CD_INV_AMT',
 'OI_TERMS',
 'OI_REF_NBR',
 'OI_ADDED_CODE',
 'OI_DPD',
 'OI_SHIPTO',
 'OI_ORIG_ORDER',
 'OI_ORIG_INVOICE',
 'OI_CUST_ORDER_1',
 'OI_CUST_ORDER_2',
 'OI_BILL_OF_LADING',
 'OI_CAR_NUMBER',
 'OI_SALESMAN',
 'OI_LOCATION',
 'OI_BATCH_NBR',
 'OI_ITEM_FROM',
 'OI_ORIG_COPC',
 'OI_TRAFFIC_DESC',
 'OI_TRAFFIC_CODE',
 'OI_ARRIVAL_DATE',
 'OI_CRA_CODE',
 'OI_CO',
 'OI_PRODUCT_CODE',
 'OI_PRODUCT_DESC',
 'OI_PRODUCT_TYPE',
 'OI_PRODUCT_GROUP',
 'OI_GETPAID_TRANS_NBR']

In [ ]:
df = adm_table_reader.read_source_table("ibmdb2_opencurr_currency")
df.dtypes

[('CURR_ID', 'int'),
 ('CURR_CODE', 'string'),
 ('CURR_DESC', 'string'),
 ('CURR_ACTIVE', 'string'),
 ('CURR_UPDATE_BY_EMPL_NO', 'string'),
 ('CURR_LAST_UPDATE', 'timestamp')]

In [82]:

df = adm_table_reader.read_source_table("ibmdb2_cop_cross_reference")
df.dtypes

[('DERIVEDKEY', 'string'),
 ('XRFC_ID', 'int'),
 ('XRFC_XTYP_ID', 'int'),
 ('XRFC_COMPANY', 'string'),
 ('XRFC_DIVISION', 'string'),
 ('XRFC_COPC', 'string'),
 ('XRFC_LOC', 'string'),
 ('PROCESSED_FILE_PATH', 'string'),
 ('PROCESSED_FILE_NAME', 'string'),
 ('INSERTED_TIMESTAMP', 'timestamp'),
 ('INSERTED_ON', 'date'),
 ('INSERTED_YEAR', 'int'),
 ('INSERTED_MONTH', 'int'),
 ('INSERTED_DAY', 'int'),
 ('INSERTED_MONTHYEAR', 'string')]

In [35]:
df = adm_table_reader.read_source_table("ibmdb2_copc_profit_center")
df.dtypes

[('PC_COPC', 'string'),
 ('PC_BASE_CURR', 'string'),
 ('PC_GRP_COMMODITY', 'string'),
 ('PC_SUB_COMMODITY', 'string'),
 ('PC_CO_NAME', 'string'),
 ('PC_PC_NAME', 'string'),
 ('PC_COPC_DELETE', 'string'),
 ('PC_COPC4', 'string')]

In [40]:
df = adm_table_reader.read_source_table("ibmdb2_credit_dist_list")
df.dtypes

[('CD_COPC', 'string'),
 ('CD_CREDIT_MGR', 'string'),
 ('CD_CR_MGR_NAME', 'string'),
 ('CD_CREDIT_REP', 'string'),
 ('CD_CR_REP_NAME', 'string'),
 ('CD_LOCATION', 'string'),
 ('CD_DORMANT', 'string'),
 ('CD_EXECUTIVE', 'string'),
 ('CD_VICE_PRESIDENT', 'string'),
 ('CD_STATE', 'string'),
 ('CD_INTL_CREDIT_MGR', 'string'),
 ('CD_INTL_CREDIT_REP', 'string'),
 ('CD_GL_CO', 'string'),
 ('CD_AR_RESP', 'string'),
 ('CD_GL_RESP', 'string'),
 ('CD_ORIG_SYSTEM', 'string'),
 ('CD_INSURE_FLAG', 'string'),
 ('CD_CSR_IDENT', 'string')]

In [41]:
df = adm_table_reader.read_source_table("ibmdb2_comd_group")
df.dtypes

[('CMDG_COMD_GRP_CD', 'string'),
 ('CMDG_DESC', 'string'),
 ('CMDG_CORP_SEGMENT_CD', 'string')]

In [42]:
df = adm_table_reader.read_source_table("ibmdb2_corp_segment")
df.dtypes

[('CSEG_SEGMENT_CD', 'string'), ('CSEG_DESC', 'string')]

In [44]:
# df = adm_table_reader.read_source_table("ibmdb2_status_code_cust")
# df.dtypes

In [50]:
df = adm_table_reader.read_source_table("ibmdb2_currencytable")
df.dtypes

[('MANDT', 'string'),
 ('KURST', 'string'),
 ('FCURR', 'string'),
 ('TCURR', 'string'),
 ('GDATU', 'string'),
 ('UKURS', 'decimal(9,5)'),
 ('FFACT', 'decimal(9,0)'),
 ('TFACT', 'decimal(9,0)')]

In [113]:
df = adm_table_reader.read_source_table("ibmdb2_bs_emea_billing_entities")
df.dtypes

INFO:adm.integration.base_integration:Starting transformations for ID: ibmdb2_bs_emea_billing_entities


[('DERIVEDKEY', 'string'),
 ('LOCATION_CODE', 'string'),
 ('ENTITY', 'string'),
 ('LOCATION_NAME_SHORT', 'string'),
 ('ENTITYBILLING', 'string'),
 ('COUNTRY', 'string'),
 ('COUNTRY_CODE', 'string'),
 ('BUSINESS_SEGMENT', 'string'),
 ('ERP_SYSTEM', 'string'),
 ('RESP_TEAM', 'string'),
 ('PROCESSED_FILE_PATH', 'string'),
 ('PROCESSED_FILE_NAME', 'string'),
 ('INSERTED_TIMESTAMP', 'timestamp'),
 ('INSERTED_ON', 'date'),
 ('INSERTED_YEAR', 'int'),
 ('INSERTED_MONTH', 'int'),
 ('INSERTED_DAY', 'int'),
 ('INSERTED_MONTHYEAR', 'string')]

In [116]:
df.count()

43

In [115]:
display(df.groupBy("LOCATION_CODE").count())

,LOCATION_CODE,count
0,MX4,1
1,UT6,1
2,5KV,1
3,D28,1
4,PH9,1
5,A2K,1
6,E9P,1
7,BUE,1
8,G9U,1
9,UT7,1


In [55]:
# df = adm_table_reader.read_source_table("ibmdb2_sales_orders_invoice_ibmdb2_table_name")
# df.dtypes

In [93]:
df = transformer2.apply_joins("combine_1")

INFO:adm.integration.base_integration:Applying joins for combine_id: combine_1
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_order_detail
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_customer_ledger


INFO:adm.integration.base_integration:Columns in cl: ['RPDOC', 'RPDCT', 'RPKCO', 'RPSDOC', 'RPLNID', 'payment_term_code', 'currency_code', 'RPBCRC', 'RPAG', 'RPACR', 'RPDDJ']
INFO:adm.integration.base_integration:Duplicate column detected and skipped: rr.requisition_id


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `ttf`.`requisition_id` cannot be resolved. Did you mean one of the following? [`od`.`user_id`, `od`.`document_type`, `cl`.`RPACR`, `cl`.`RPAG`, `cl`.`RPBCRC`]. SQLSTATE: 42703;
'Project ['ttf.requisition_id AS requisition_id#45564, 'ttf.job_type AS job_type#45565, 'ttf.pay_grade_band AS pay_grade_band#45566, 'ttf.hiring_manager_name AS hiring_manager_name#45567, 'ttf.recruiter_name AS recruiter_name#45568, 'ttf.organizational_level_2 AS organizational_level_2#45569, 'ttf.organizational_level_3 AS organizational_level_3#45570, 'ttf.business_id AS business_id#45571, 'rr.division AS division#45572, 'rr.old_reason_for_hire AS old_reason_for_hire#45573, 'rr.ec_reason_for_hire AS ec_reason_for_hire#45574, 'rr.additional_replacement_justification AS additional_replacement_justification#45575, 'rr.college_relations_requisition AS college_relations_requisition#45576, 'rr.fte_number_of_posi

In [73]:
df = transformer.apply_transformations("ffjt", df)

INFO:adm.integration.base_integration:Starting transformations for ID: ffjt


In [ ]:
writer.write_table("silver_globlal_hr_request_poc", df)

INFO:adm.integration.base_integration:Getting schema from schema file use_cases/h2r/h2r_silver_head_count_poc_schema.json for sandbox.de_ref_silver_mikesturwoldcontractor.silver_head_count_poc


INFO:adm.integration.base_integration:Number of records written to table sandbox.de_ref_silver_mikesturwoldcontractor.silver_head_count_poc: 200


In [41]:
df = adm_table_reader.read_source_table("req_report")
#df_payroll = adm_table_reader.read_source_table("payroll")
#df_employee = adm_table_reader.read_source_table("employee")
#df_employee = transformer.apply_transformations("employee_new", df_employee)
#df_employee = transformer.apply_transformations("employee_expr", df_employee)

#df = transformer2.apply_joins("emp_payroll")
# df = transformer.apply_transformations("employee", df)
#df = transformer.apply_transformations("employee_expr_new", df_employee)
#df = df.select(tgt_cols)
#writer = AdmTableWriter("scratch/mike/h2r_silver_head_count_poc.yml")
#writer.write_table("silver_head_count_poc", df)

INFO:adm.integration.base_integration:Starting transformations for ID: req_report


In [0]:
#Source/adm_integration_framework/scratch/mike/h2r_head_count_poc.yml
#adm_table_reader = AdmTableReader("scratch/mike/h2r_silver_head_count_poc.yml")
#df_payroll = adm_table_reader.read_source_table("payroll")
#df_employee= adm_table_reader.read_source_table("employee")
#display(df_employee.select("bus_id","bus_id_trim"))
#display(result_df.selectExpr("* except(Emp_ID, First_Name)"))

In [0]:
#transformer = AdmBaseDataTransformer("scratch/mike/h2r_head_count_poc.yml")
#tx = "rn"

#result_df = transformer.apply_transformations("rn", result_df)
#result_df = transformer.apply_transformations("cast_cols", result_df)
#display(result_df)
#display(transformed_df.selectExpr("*", "cast(overtime_pay as decimal(10,2)) as overtime_pay"))
# why do i need to setup logger here?
# fixed by adding self.logger to adm_base_data_transformer

# column expressions duplicate columns if * specified

#transformer = AdmDataTransformer("scratch/mike/h2r_silver_head_count_poc.yml")
#result_df = transformer.apply_joins("emp_payroll")




In [0]:
display(result_df)

In [0]:
df = result_df.transform(p2c_custom_tranformatin, Col, cole3)

In [0]:
writer = AdmTableWriter("scratch/mike/h2r_silver_head_count_poc.yml")
# why does writer create the table?
# why does it need a schema file?
writer.write_table("silver_head_count_poc", result_df)

In [0]:
for x in spark.table("sandbox.h2r_dbx_poc.ec_employee").columns:
    print(x)

In [0]:
for x in spark.table("sandbox.h2r_dbx_poc.ec_payroll").columns:
    print(x)